In [12]:
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Ninguna'}")

GPU disponible: True
GPU: Tesla T4


In [13]:
!pip install nibabel numpy scipy scikit-image matplotlib \
             ollama sanitext datasets -q

In [14]:
# Clonar directamente tu repo SGEM con todos los módulos incluidos
!git clone https://github.com/EstebanArenas1/SGEM.git
%cd SGEM
import sys
sys.path.insert(0, ".")
print("Repo SGEM clonado correctamente")

Cloning into 'SGEM'...
remote: Enumerating objects: 997, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 997 (delta 2), reused 12 (delta 2), pack-reused 985 (from 1)
Receiving objects: 100% (997/997), 740.98 MiB | 6.16 MiB/s, done.
Resolving deltas: 100% (546/546), done.
/content/SGEM/SGEM
Repo SGEM clonado correctamente


In [15]:
# Copiar módulos SGEM directamente (sin necesitar GitHub)
sgem_clasificar = '''
import logging
import nibabel as nib
import numpy as np

logger = logging.getLogger(__name__)

CISTERNAS_BASALES = [2, 3, 4, 5, 7]
CISTERNAS_SYLVIANAS = [13]
CISTERNAS_OTRAS = [1, 6, 8, 9, 10, 11, 12]
LABELS_CLASIFICACION = CISTERNAS_BASALES
UMBRAL_COMPRIMIDAS = 0.5
UMBRAL_PARCIAL = 0.2

def clasificar_cisternas(cisterna_registrada_path, tumor_mask_path):
    seg_nii = nib.load(cisterna_registrada_path)
    tumor_nii = nib.load(tumor_mask_path)
    cisterna = seg_nii.get_fdata().astype(int)
    tumor = tumor_nii.get_fdata().astype(int)
    tumor_binario = tumor > 0
    mascara_basales = np.isin(cisterna, LABELS_CLASIFICACION)
    volumen_total = int(np.sum(mascara_basales))
    overlap_total = int(np.sum(mascara_basales & tumor_binario))
    compression_ratio = overlap_total / volumen_total if volumen_total > 0 else 0.0
    if compression_ratio >= UMBRAL_COMPRIMIDAS:
        estado = "comprimidas"
    elif compression_ratio >= UMBRAL_PARCIAL:
        estado = "parcialmente comprimidas"
    else:
        estado = "preservadas"
    return {
        "cisterns_status": estado,
        "compression_ratio": round(float(compression_ratio), 3),
        "cistern_overlap_voxels": overlap_total,
        "cistern_total_voxels": volumen_total,
    }

def texto_reporte(resultado):
    estado = resultado["cisterns_status"]
    cr = resultado["compression_ratio"]
    if estado == "comprimidas":
        return f"Las cisternas basales se encuentran comprimidas (indice de compresion: {cr:.2f}), sugestivo de efecto de masa significativo."
    elif estado == "parcialmente comprimidas":
        return f"Las cisternas basales presentan compresion parcial (indice de compresion: {cr:.2f}), compatible con efecto de masa moderado."
    else:
        return f"Las cisternas basales se encuentran preservadas (indice de compresion: {cr:.2f})."
'''

sgem_mls = '''
import logging
import nibabel as nib
import numpy as np

logger = logging.getLogger(__name__)
SP_LABELS = [10, 49]
UMBRAL_SIGNIFICATIVO = 5.0
UMBRAL_CRITICO = 10.0

def medir_mls_septum(anat_seg_path):
    seg_nii = nib.load(anat_seg_path)
    seg = seg_nii.get_fdata().astype(int)
    voxel_size = float(seg_nii.header.get_zooms()[0])
    sp_mask = np.isin(seg, SP_LABELS)
    if np.sum(sp_mask) == 0:
        raise ValueError("No se detecto septum pellucidum.")
    sp_voxels = np.argwhere(sp_mask)
    septum_centroid_x = float(np.mean(sp_voxels[:, 0]))
    brain_mask = seg > 0
    brain_voxels = np.argwhere(brain_mask)
    x_min = float(brain_voxels[:, 0].min())
    x_max = float(brain_voxels[:, 0].max())
    brain_center_x = (x_min + x_max) / 2.0
    desplazamiento_voxels = septum_centroid_x - brain_center_x
    mls_mm = abs(desplazamiento_voxels) * voxel_size
    direccion = "izquierda->derecha" if desplazamiento_voxels > 0 else "derecha->izquierda"
    if mls_mm < 3.0:
        categoria = "ausente"
    elif mls_mm < 5.0:
        categoria = "leve"
    elif mls_mm < 10.0:
        categoria = "moderado"
    else:
        categoria = "severo"
    return {
        "mls_septum_mm": round(mls_mm, 2),
        "mls_direccion": direccion,
        "mls_significativo": mls_mm >= UMBRAL_SIGNIFICATIVO,
        "mls_critico": mls_mm >= UMBRAL_CRITICO,
        "mls_categoria": categoria,
    }
'''

# Guardar módulos en el repo clonado
with open("btreport/utils/clasificar_cisternas.py", "w") as f:
    f.write(sgem_clasificar)

with open("btreport/utils/mls_septum.py", "w") as f:
    f.write(sgem_mls)

print("Módulos SGEM copiados correctamente")

Módulos SGEM copiados correctamente


In [16]:
import os

# Descargar caso público de BraTS 2023
!pip install gdown -q
import gdown

# Caso de ejemplo (BraTS-GLI-00000-000)
os.makedirs("data/BraTS-GLI-00000-000", exist_ok=True)

# Alternativa: usar el dataset de HuggingFace para ver features sin imágenes
import json
with open("btreport_brats23.json") as f:
    data = json.load(f)

case_id = list(data.keys())[0]
print(f"Caso: {case_id}")
print(f"Reporte existente:\n{data[case_id]['Predicted Report (llama3:70b)'][:500]}")

Caso: BraTS-GLI-00000-000
Reporte existente:
FINDINGS:

MASS EFFECT & VENTRICLES: There is a minimal left-to-right midline shift of approximately 4 mm at the level of the septum pellucidum. The left lateral ventricle is effaced in its anterior horn.

BRAIN/ENHANCEMENT: Within the left frontal lobe, there is a solitary, predominantly enhancing mass measuring 4.3 x 4.3 x 6.6 cm. The enhancement quality is marked, with a thick (>3mm) enhancing margin. There is cortical involvement and deep white matter invasion present. Ependymal invasion is 


In [17]:
# Instalar dependencia y luego Ollama
!sudo apt-get install -y zstd -q
!curl -fsSL https://ollama.com/install.sh | sh

# Arrancar servidor en background
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

# Descargar llama3:8b
!ollama pull llama3:8b

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 67 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [18]:
import sys
sys.path.insert(0, ".")

import numpy as np
import nibabel as nib
from btreport.utils.clasificar_cisternas import clasificar_cisternas, texto_reporte

# Test con atlas real
atlas = nib.load("btreport/utils/Cistern_Segmentations.nii.gz")
data_atlas = atlas.get_fdata().astype(np.int16)
affine = atlas.affine

# Tumor simulado
tumor = np.zeros_like(data_atlas, dtype=np.int16)
tumor[70:120, 90:140, 70:120] = 2
nib.save(nib.Nifti1Image(tumor, affine), "/tmp/tumor_test.nii.gz")
nib.save(nib.Nifti1Image(data_atlas, affine), "/tmp/cisternas_test.nii.gz")

resultado = clasificar_cisternas("/tmp/cisternas_test.nii.gz", "/tmp/tumor_test.nii.gz")
print(texto_reporte(resultado))

Las cisternas basales se encuentran preservadas (indice de compresion: 0.09).


In [19]:
# Crear el módulo del prompt en español directamente en Colab
sgem_prompt = '''
import json
import ollama

EJEMPLOS_HALLAZGOS = """
1.
HALLAZGOS:
EFECTO DE MASA Y VENTRICULOS: Desplazamiento de la linea media de aproximadamente 8 mm
hacia la izquierda a nivel del septum pellucidum. Las cisternas basales se encuentran
comprimidas. Efacement del asta frontal del ventriculo lateral derecho.
LESION Y REALCE: En el lobulo frontal derecho lesion con realce en anillo que mide
3.4 x 3.2 x 3.5 cm. Area de hipointensidad central compatible con necrosis.

2.
HALLAZGOS:
EFECTO DE MASA Y VENTRICULOS: Desplazamiento de linea media de aproximadamente 11 mm
de derecha a izquierda medido a nivel del septum pellucidum. Las cisternas basales
estan parcialmente comprimidas.
LESION Y REALCE: Lesion en lobulo parieto-occipital derecho que mide 5.4 x 4.1 x 3.5 cm
con realce periferico nodular. Area de senal FLAIR perilesional que cruza el esplenio.

3.
HALLAZGOS:
EFECTO DE MASA Y VENTRICULOS: Efacement de los astas anteriores de los ventriculos
laterales. Desplazamiento de linea media de aproximadamente 5 mm. Cisternas basales preservadas.
LESION Y REALCE: Lesion de gran tamano en lobulo frontal paramediano izquierdo que
cruza el cuerpo calloso. Area necrotica central que mide hasta 2.5 cm.
"""

PLANTILLA = """
Eres un radiologo generando un reporte clinico de RM cerebral en espanol.

Ejemplos de reportes reales:
{ejemplos}

Genera una seccion de HALLAZGOS usando UNICAMENTE los metadatos proporcionados.

INSTRUCCIONES:
- Escribe en espanol clinico formal.
- Secciones: EFECTO DE MASA Y VENTRICULOS, y LESION Y REALCE.
- Comenta el MLS usando mls_septum_mm y mls_direccion.
- Comenta el estado de las cisternas usando cisterns_status.
- Selecciona los 7-10 hallazgos mas relevantes.
- NO inventes informacion no respaldada por los metadatos.

METADATOS (paciente {subject_id}):
{metadata_json}

Escribe ahora la seccion de HALLAZGOS en espanol clinico formal.
"""

def generar_reporte_es(subject_id, metadata, image_path=None, model="llama3:8b"):
    prompt = PLANTILLA.format(
        ejemplos=EJEMPLOS_HALLAZGOS,
        subject_id=subject_id,
        metadata_json=json.dumps(metadata, indent=2, ensure_ascii=False),
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]
'''

with open("btreport/llm_report_generation/ollama_report_gen_sgem.py", "w") as f:
    f.write(sgem_prompt)

print("Módulo prompt español creado")

Módulo prompt español creado


In [20]:
from btreport.llm_report_generation.ollama_report_gen_sgem import generar_reporte_es

# Metadata de ejemplo (en producción viene del pipeline completo)
metadata_ejemplo = {
    "Tumor Location": "left frontal lobe",
    "Side of Tumor Epicenter": "left",
    "Proportion Enhancing": 57.2,
    "Proportion Necrosis": 20.1,
    "Proportion of Oedema": 22.7,
    "Lesion Sizes APxTVxCC (cm)": "4.3 x 4.3 x 6.6",
    "midline_shift_present": "Yes",
    "max_shift_mm": 4.1,
    "level_max_shift": "septum pellucidum",
    # Variables SGEM
    "cisterns_status": resultado["cisterns_status"],
    "compression_ratio": resultado["compression_ratio"],
    "mls_septum_mm": 4.1,
    "mls_direccion": "izquierda→derecha",
    "mls_categoria": "leve",
}

reporte = generar_reporte_es(
    subject_id="BraTS-GLI-00000-000",
    metadata=metadata_ejemplo,
    model="llama3:8b"
)
print(reporte)

HALLAZGOS:

EFECTO DE MASA Y VENTRICULOS:
Presenta un desplazamiento de la línea media de 4.1 mm hacia la izquierda a nivel del septum pellucidum, asociado a una leve desviación de la línea media en dirección izquierda→derecha.

EFECTO DE VENTRICULOS:
No se observa notable desplazamiento o compresión de los ventriculos laterales.

LESION Y REALCE:
Presenta una lesión en el lóbulo frontal izquierdo que mide 4.3 x 4.3 x 6.6 cm. No se observa necrosis central, aunque se aprecia una zona de edema asociada.

Notar que no se han observado lesiones significativas en el lobulo frontal derecho ni en la parte posterior del cerebro.


In [22]:
from google.colab import files
import os

os.makedirs("data/HPTU/HPTU-001", exist_ok=True)

print("Selecciona el archivo: HPTU-001-t1n.nii.gz")
print("(el que pesa ~1.8MB, NO el .json)")
uploaded = files.upload()

for filename in uploaded.keys():
    dest = f"data/HPTU/HPTU-001/{filename}"
    os.rename(filename, dest)
    print(f"✅ Subido: {filename} → {dest}")

Selecciona el archivo: HPTU-001-t1n.nii.gz
(el que pesa ~1.8MB, NO el .json)


Saving HPTU-001-t1n.nii.gz to HPTU-001-t1n.nii (1).gz
✅ Subido: HPTU-001-t1n.nii (1).gz → data/HPTU/HPTU-001/HPTU-001-t1n.nii (1).gz


In [23]:
# Instalar Miniconda con Python 3.10 en Colab
!wget -qO /tmp/miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py310_23.5.2-0-Linux-x86_64.sh
!bash /tmp/miniconda.sh -b -p /opt/miniconda
print("Miniconda instalado")

PREFIX=/opt/miniconda
Unpacking payload ...
                                                                              
Installing base environment...





Preparing transaction: - \ | / done
Executing transaction: \ | / - \ | / - \ | / - \ | / - \ | / - \ | / done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/miniconda
Miniconda instalado


In [24]:
!/opt/miniconda/bin/conda create -n synthseg python=3.10 -y -q
!/opt/miniconda/envs/synthseg/bin/pip install tensorflow==2.13.0 nibabel numpy==1.23.5 scikit-image -q
print("Entorno SynthSeg listo")

Solving environment: ...working... done

## Package Plan ##

  environment location: /opt/miniconda/envs/synthseg

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-5.1          |           52_gnu           7 KB
    bzip2-1.0.8                |       h5eee18b_6         262 KB
    ca-certificates-2026.7.16  |       h06a4308_0         106 KB
    ld_impl_linux-64-2.44      |       h9e0c5a2_3         725 KB
    libexpat-2.8.2             |       h7354ed3_1         126 KB
    libffi-3.4.8               |       h06d3fd0_3         137 KB
    libgcc-15.2.0              |       h69a1729_8         803 KB
    libgcc-ng-15.2.0           |       h166f726_8          28 KB
    libnsl-2.0.0               |       h5eee18b_0          31 KB
    libstdcxx-15.2.0           |       h39759b7_8         3.7 MB
    libxcb-1.17.0              |       h9b100f

In [29]:
!/opt/miniconda/envs/synthseg/bin/python /content/SGEM/SGEM/SynthSeg/scripts/commands/SynthSeg_predict.py \
    --i /content/SGEM/data/HPTU/HPTU-001/HPTU-001-t1n.nii.gz \
    --o /content/SGEM/data/HPTU/HPTU-001/HPTU-001-synthseg.nii.gz \
    --v1 \
    --vol /content/SGEM/data/HPTU/HPTU-001/HPTU-001-volumes.csv

2026-08-25 13:28:43.780268: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-25 13:28:44.310967: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-25 13:28:44.311551: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-25 13:28:45.208542: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT

SynthSeg 1.0

using 1 thread
Traceback (most recent call last):
  File "/content/SGEM/SGEM/SynthSeg/scripts/commands/SynthSeg_predict.py", line 113, in <module>
    predict(path_images=args['i'],
  File "/content/SGEM/SGEM/SynthSeg/SynthSeg/predict_synthseg.py", line 74, in predict


In [30]:
!find /content -name "*.nii.gz" 2>/dev/null

/content/SGEM/HPTU-001-t1c.nii.gz
/content/SGEM/SGEM/HPTU-001-t1n.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_20.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_15.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_09.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_01.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_10.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_17.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_11.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_04.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_16.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_05.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_08.nii.gz
/content/SGEM/SGEM/SynthSeg/data/training_label_maps/training_seg_14.nii.gz
/content/SGEM/S

In [31]:
!/opt/miniconda/envs/synthseg/bin/python /content/SGEM/SGEM/SynthSeg/scripts/commands/SynthSeg_predict.py \
    --i /content/SGEM/SGEM/HPTU-001-t1n.nii.gz \
    --o /content/SGEM/SGEM/HPTU-001-synthseg.nii.gz \
    --v1 \
    --vol /content/SGEM/SGEM/HPTU-001-volumes.csv

2026-08-25 13:29:39.559485: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-25 13:29:39.613041: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-25 13:29:39.613609: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-25 13:29:40.402790: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT

SynthSeg 1.0

using 1 thread
2026-08-25 13:29:42.762365: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https:/

In [33]:
import nibabel as nib
import numpy as np
from scipy.ndimage import zoom

# Cargar atlas y segmentación
atlas_nii = nib.load(ATLAS_CISTERNAS)
seg_nii = nib.load(SYNTHSEG)

atlas = atlas_nii.get_fdata().astype(np.int16)
seg = seg_nii.get_fdata().astype(np.int16)

# Calcular factores de zoom para remuestrear atlas al tamaño de synthseg
factores = [seg.shape[i] / atlas.shape[i] for i in range(3)]
atlas_resampled = zoom(atlas, factores, order=0)  # order=0 para segmentaciones

print(f"Atlas original: {atlas.shape}")
print(f"SynthSeg: {seg.shape}")
print(f"Atlas remuestreado: {atlas_resampled.shape}")

# Guardar atlas remuestreado
atlas_res_path = "/tmp/atlas_cisternas_resampled.nii.gz"
nib.save(nib.Nifti1Image(atlas_resampled, seg_nii.affine), atlas_res_path)

# Tumor vacío del mismo tamaño
tumor_vacio = np.zeros(seg.shape, dtype=np.int16)
nib.save(nib.Nifti1Image(tumor_vacio, seg_nii.affine), "/tmp/tumor_vacio.nii.gz")

# Clasificar cisternas
cisternas = clasificar_cisternas(atlas_res_path, "/tmp/tumor_vacio.nii.gz")
print(f"\nEstado: {cisternas['cisterns_status']}")
print(f"CR: {cisternas['compression_ratio']}")
print(f"\n{texto_reporte(cisternas)}")

Atlas original: (182, 218, 182)
SynthSeg: (241, 240, 164)
Atlas remuestreado: (241, 240, 164)

Estado: preservadas
CR: 0.0

Las cisternas basales se encuentran preservadas (indice de compresion: 0.00).


In [34]:
from btreport.llm_report_generation.ollama_report_gen_sgem import generar_reporte_es

metadata_hptu001 = {
    "Tumor Location": "pendiente de segmentación",
    "Side of Tumor Epicenter": "pendiente",
    "Lesion Sizes APxTVxCC (cm)": "pendiente de segmentación",
    "midline_shift_present": "No",
    "max_shift_mm": 1.12,
    "level_max_shift": "septum pellucidum",
    # Módulos SGEM — valores reales
    "mls_septum_mm": mls["mls_septum_mm"],
    "mls_direccion": mls["mls_direccion"],
    "mls_categoria": mls["mls_categoria"],
    "cisterns_status": cisternas["cisterns_status"],
    "compression_ratio": cisternas["compression_ratio"],
}

print("Generando reporte en español con llama3:8b...")
reporte = generar_reporte_es(
    subject_id="HPTU-001",
    metadata=metadata_hptu001,
    model="llama3:8b"
)
print("\n=== REPORTE SGEM — HPTU-001 ===")
print(reporte)

Generando reporte en español con llama3:8b...

=== REPORTE SGEM — HPTU-001 ===
HALLAZGOS:

EFECTO DE MASA Y VENTRICULOS:
El estudio de RM cerebral revela un desplazamiento de la línea media de 1.12 mm hacia la derecha a nivel del septum pellucidum, indicando un efecto de masa cerebral. No se observa efecto significativo sobre los ventriculos.

LESION Y REALCE:
No se identifican lesiones o realces relevantes en el estudio de RM cerebral.
